# Phase 7: Prior Case Retrieval - Legal Transformer with LoRMA

## Task
Prior case retrieval: for each query case, find the earlier cases it cites from a pool of candidate cases. The 7 best candidates are returned per query and scored with Micro-F1@7.

Model: the pre-trained Legal Transformer fine-tuned with LoRMA (low-rank multiplicative updates). Its scores on the train split are also saved so they can be used as a teacher for distillation.

## Input
- Phase-7 data (queries and candidates for train / dev / test); downloaded from Hugging Face if not attached (needs HF_TOKEN).
- Legal Transformer checkpoint and tokenizer (found automatically).

## Flow of the notebook (from input to output)
1. Write the model code file.
2. Import libraries and set up the file finder.
3. Load the Legal Transformer and set the LoRMA settings.
4. Load the queries and candidate cases of the train, dev and test splits.
5. Compute BM25 and TF-IDF scores between every query and every candidate.
6. Define the Micro-F1@7 metric and the score mixing (z-scores, hubness correction, balancing).
7. Define the fine-tuning methods (LoRMA, adapters, full) with memory-saving blocks.
8. Define how documents become token chunks and padded batches.
9. Define how documents become vectors (mean over tokens and chunks) and cosine similarity scores.
10. Define the training loss: contrastive (each query must pick its cited case among hard negatives), plus distillation if a teacher is given.
11. Define the training loop (cosine learning rate, dev check every epoch, keep the best epoch).
12. Tokenize everything, build positive pairs and hard negatives from BM25 / TF-IDF.
13. Fine-tune the model.
14. Compute and save the model scores for all splits.
15. Choose the model weight, BM25 weight and post-processing on dev and write the 4 best submissions.
16. Zip all outputs.

## Output
- 4 submission zips (dev score and settings in the file name).
- Saved score matrices (a file ending in _scores.npz) and a results JSON.
- One zip with all outputs.

In [1]:
%%writefile arch_legal_transformer.py
# Legal Transformer model code (saved as arch_legal_transformer.py)
import json
import math
import os
import re as _re
import shutil
import tempfile
import time as _time
import zipfile
from collections import Counter, defaultdict as _defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

ARCH = "legal_transformer"
TRAINED_SEQ_TOKENS = 640
NOTEBOOK_ROPE_LEN = TRAINED_SEQ_TOKENS + 8

# It maps every byte to a printable character for the byte-level tokenizer and returns a dictionary from byte value to character.
def _bytes_to_unicode():

    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("\xa1"), ord("\xac") + 1)) + \
         list(range(ord("\xae"), ord("\xff") + 1))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return dict(zip(bs, [chr(c) for c in cs]))

_BYTE_ENCODER = _bytes_to_unicode()
_BYTE_DECODER = {v: k for k, v in _BYTE_ENCODER.items()}

_PRETOK_RE = _re.compile(r"'s|'t|'re|'ve|'m|'ll|'d| ?[A-Za-z]+| ?[0-9]+| ?[^\sA-Za-z0-9]+|\s+")

class LegalByteBPETokenizer:

    PAD_TOKEN = "<pad>"
    EOS_TOKEN = "<|endoftext|>"

    # It creates an empty tokenizer and returns nothing.
    def __init__(self):
        self.vocab = {}
        self.id_to_token = {}
        self.merge_rank = {}
        self.pad_token_id = None
        self.eos_token_id = None
        self.padding_side = "right"
        self._cache = {}

    # It returns the vocabulary size.
    def __len__(self):
        return len(self.vocab)

    # It takes texts, target_vocab_size (target vocabulary size), sample_chars_per_doc (characters used per document) and verbose (print progress) as input, learns the BPE merges and vocabulary from the training texts and returns the trained tokenizer.
    def train(self, texts, target_vocab_size, sample_chars_per_doc=None, verbose=True):

        t0 = _time.time()
        word_freq = Counter()
        for text in texts:
            if sample_chars_per_doc is not None:
                text = text[:sample_chars_per_doc]
            for m in _PRETOK_RE.finditer(text):
                sym = tuple(_BYTE_ENCODER[byte] for byte in m.group().encode("utf-8"))
                word_freq[sym] += 1

        self.vocab = {self.PAD_TOKEN: 0, self.EOS_TOKEN: 1}
        for i in range(256):
            self.vocab[_BYTE_ENCODER[i]] = len(self.vocab)
        self.pad_token_id = self.vocab[self.PAD_TOKEN]
        self.eos_token_id = self.vocab[self.EOS_TOKEN]

        words = {i: list(w) for i, w in enumerate(word_freq)}
        freqs = {i: c for i, (w, c) in enumerate(word_freq.items())}
        pair_counts = Counter()
        pair_to_words = _defaultdict(set)
        for wid, syms in words.items():
            f = freqs[wid]
            for i in range(len(syms) - 1):
                p = (syms[i], syms[i + 1])
                pair_counts[p] += f
                pair_to_words[p].add(wid)

        n_merges_needed = target_vocab_size - len(self.vocab)
        n_merges_done = 0
        while n_merges_done < n_merges_needed and pair_counts:
            max_freq = max(pair_counts.values())

            best = min(p for p, c in pair_counts.items() if c == max_freq)
            new_symbol = best[0] + best[1]
            self.merge_rank[best] = n_merges_done
            self.vocab[new_symbol] = len(self.vocab)

            for wid in list(pair_to_words.get(best, ())):
                syms, f = words[wid], freqs[wid]
                for i in range(len(syms) - 1):
                    p = (syms[i], syms[i + 1])
                    pair_counts[p] -= f
                    if pair_counts[p] <= 0:
                        pair_counts.pop(p, None)
                    pair_to_words[p].discard(wid)
                merged, i = [], 0
                while i < len(syms):
                    if i < len(syms) - 1 and syms[i] == best[0] and syms[i + 1] == best[1]:
                        merged.append(new_symbol); i += 2
                    else:
                        merged.append(syms[i]); i += 1
                words[wid] = merged
                for i in range(len(merged) - 1):
                    p = (merged[i], merged[i + 1])
                    pair_counts[p] += f
                    pair_to_words[p].add(wid)
            pair_counts.pop(best, None)
            pair_to_words.pop(best, None)
            n_merges_done += 1

        self.id_to_token = {v: k for k, v in self.vocab.items()}
        if verbose:
            print(f"    BPE training: {n_merges_done} merges learned (target {n_merges_needed}) in "
                  f"{_time.time() - t0:.1f}s over {len(word_freq)} unique pretoken types "
                  f"-> final vocab size = {len(self.vocab)}")
        return self

    # It takes symbol_tuple (word as symbols) as input, applies the learned merges to one word and returns the word split into tokens (as a string).
    def _bpe(self, symbol_tuple):

        cached = self._cache.get(symbol_tuple)
        if cached is not None:
            return cached
        word = list(symbol_tuple)
        while len(word) > 1:
            best_rank, best_i = None, -1
            for i in range(len(word) - 1):
                r = self.merge_rank.get((word[i], word[i + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_i = r, i
            if best_rank is None:
                break
            word = word[:best_i] + [word[best_i] + word[best_i + 1]] + word[best_i + 2:]
        self._cache[symbol_tuple] = word
        return word

    # It takes text as input, converts text into a list of token ids and returns a list of token ids.
    def encode(self, text):
        ids = []
        for m in _PRETOK_RE.finditer(text):
            sym = tuple(_BYTE_ENCODER[byte] for byte in m.group().encode("utf-8"))
            for piece in self._bpe(sym):
                ids.append(self.vocab[piece])
        return ids

    # It takes ids (token ids) and skip_special_tokens (drop special tokens) as input, converts token ids back into text and returns text.
    def decode(self, ids, skip_special_tokens=True):

        special = {self.pad_token_id, self.eos_token_id}
        pieces = [self.id_to_token.get(int(i), "") for i in ids
                  if not (skip_special_tokens and int(i) in special)]
        raw = bytes(_BYTE_DECODER[c] for c in "".join(pieces) if c in _BYTE_DECODER)
        return raw.decode("utf-8", errors="replace")

    # It takes path (file path) as input, saves the tokenizer to a JSON file and returns nothing.
    def save(self, path):

        data = {
            "vocab": self.vocab,
            "merges": [list(pair) for pair, _rank in
                       sorted(self.merge_rank.items(), key=lambda kv: kv[1])],
            "pad_token_id": self.pad_token_id,
            "eos_token_id": self.eos_token_id,
            "padding_side": self.padding_side,
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f)

    # It takes path (file path) as input, loads a tokenizer from a JSON file and returns a tokenizer.
    @classmethod
    def load(cls, path):

        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls()
        tok.vocab = data["vocab"]
        tok.id_to_token = {int(v): k for k, v in tok.vocab.items()}
        tok.merge_rank = {tuple(pair): i for i, pair in enumerate(data["merges"])}
        tok.pad_token_id = data["pad_token_id"]
        tok.eos_token_id = data["eos_token_id"]
        tok.padding_side = data.get("padding_side", "right")
        return tok

    # It takes text, max_length (max length), truncation (cut long inputs), padding (padding option) and return_tensors (return tensors) as input, tokenizes one text or a list of texts, with optional cutting and padding and returns a dictionary with input_ids and attention_mask.
    def __call__(self, text, max_length=None, truncation=False, padding=None, return_tensors=None):
        is_batch = isinstance(text, (list, tuple))
        texts = list(text) if is_batch else [text]
        all_ids = [self.encode(t) for t in texts]
        if truncation and max_length is not None:
            all_ids = [ids[:max_length] for ids in all_ids]
        lens = [len(ids) for ids in all_ids]

        if padding == "max_length" and max_length is not None:
            pad_to = max_length
        elif padding is True or padding == "longest":
            pad_to = max(lens) if lens else 0
        else:
            pad_to = None

        if pad_to is not None:
            if self.padding_side == "right":
                attn = [[1] * L + [0] * (pad_to - L) for L in lens]
                all_ids = [ids + [self.pad_token_id] * (pad_to - len(ids)) for ids in all_ids]
            else:
                attn = [[0] * (pad_to - L) + [1] * L for L in lens]
                all_ids = [[self.pad_token_id] * (pad_to - len(ids)) + ids for ids in all_ids]
        else:
            attn = [[1] * L for L in lens]

        if return_tensors == "pt":
            return {"input_ids": torch.tensor(all_ids, dtype=torch.long),
                    "attention_mask": torch.tensor(attn, dtype=torch.long)}
        return {"input_ids": all_ids if is_batch else all_ids[0],
                "attention_mask": attn if is_batch else attn[0]}

# It takes head_dim (size of one head), max_len (max length), device and base (rotary base value) as input, pre-computes the sine and cosine tables for rotary positions and returns (cos table, sin table).
def build_rope_cache(head_dim, max_len, device, base=10000.0):

    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(max_len, device=device).float()
    freqs = torch.outer(t, inv_freq)
    emb = torch.cat([freqs, freqs], dim=-1)
    return emb.cos(), emb.sin()

# It takes x (input tensor) as input, rotates half of the vector, used by rotary positions and returns rotated tensor.
def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)

# It takes q (queries), k (keys), cos (rotary cos table), sin (rotary sin table) and pos_offset (position offset) as input, adds rotary position information to queries and keys and returns (rotated queries, rotated keys).
def apply_rope(q, k, cos, sin, pos_offset):

    T = q.size(2)
    cos_t = cos[pos_offset:pos_offset + T].unsqueeze(0).unsqueeze(0)
    sin_t = sin[pos_offset:pos_offset + T].unsqueeze(0).unsqueeze(0)
    q_rot = q * cos_t + rotate_half(q) * sin_t
    k_rot = k * cos_t + rotate_half(k) * sin_t
    return q_rot, k_rot

USE_SDPA = True

class CausalSelfAttention(nn.Module):

    # It takes dim (model size), n_heads (number of heads) and dropout (dropout rate) as input, creates the attention layer and returns nothing.
    def __init__(self, dim, n_heads, dropout=0.1):
        super().__init__()
        assert dim % n_heads == 0, "dim must be divisible by n_heads"
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.wq = nn.Linear(dim, dim)
        self.wk = nn.Linear(dim, dim)
        self.wv = nn.Linear(dim, dim)
        self.wo = nn.Linear(dim, dim)
        self.attn_drop_p = dropout
        self.resid_drop = nn.Dropout(dropout)

        self.bidirectional = False
        self.key_pad_mask = None

    # It takes x (input tensor), cos (rotary cos table), sin (rotary sin table), pos_offset (position offset) and cache (saved keys and values) as input, runs attention where each token looks only at earlier tokens and returns (output vectors, updated cache).
    def forward(self, x, cos, sin, pos_offset=0, cache=None):
        B, T, C = x.shape
        q = self.wq(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.wk(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        q, k = apply_rope(q, k, cos, sin, pos_offset)

        if cache is not None:
            past_k, past_v = cache
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)
        new_cache = (k, v)

        Tq, Tk = q.size(2), k.size(2)
        q_idx = torch.arange(Tk - Tq, Tk, device=x.device)
        k_idx = torch.arange(Tk, device=x.device)
        causal_mask = k_idx.unsqueeze(0) > q_idx.unsqueeze(1)
        if self.bidirectional:
            causal_mask = torch.zeros_like(causal_mask)

        if USE_SDPA:
            allowed = ~causal_mask
            if self.key_pad_mask is not None:
                allowed = allowed[None, None] & self.key_pad_mask[:, None, None, :]
                allowed = allowed | (k_idx.unsqueeze(0) == q_idx.unsqueeze(1))[None, None]
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed, dropout_p=self.attn_drop_p if self.training else 0.0)
        else:
            att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
            att = att.masked_fill(causal_mask, float("-inf"))
            if self.key_pad_mask is not None:
                att = att.masked_fill(~self.key_pad_mask[:, None, None, :], -1e9)
            att = F.softmax(att, dim=-1)
            att = F.dropout(att, p=self.attn_drop_p, training=self.training)
            out = att @ v
        out = out.transpose(1, 2).reshape(B, T, C)
        return self.resid_drop(self.wo(out)), new_cache

class TransformerMLP(nn.Module):

    # It takes dim (model size), mult (size multiplier) and dropout (dropout rate) as input, creates the feed-forward layer and returns nothing.
    def __init__(self, dim, mult=4, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim * mult)
        self.fc2 = nn.Linear(dim * mult, dim)
        self.drop = nn.Dropout(dropout)

    # It takes x (input tensor) as input, runs the feed-forward layer and returns output vectors.
    def forward(self, x):
        return self.drop(self.fc2(F.gelu(self.fc1(x))))

class TransformerBlock(nn.Module):

    # It takes dim (model size), n_heads (number of heads) and dropout (dropout rate) as input, creates one Transformer block and returns nothing.
    def __init__(self, dim, n_heads, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.attn = CausalSelfAttention(dim, n_heads, dropout)
        self.ln2 = nn.LayerNorm(dim)
        self.mlp = TransformerMLP(dim, dropout=dropout)

    # It takes x (input tensor), cos (rotary cos table), sin (rotary sin table), pos_offset (position offset) and cache (saved keys and values) as input, runs one Transformer block and returns (output vectors, updated cache).
    def forward(self, x, cos, sin, pos_offset=0, cache=None):
        a, new_cache = self.attn(self.ln1(x), cos, sin, pos_offset, cache)
        x = x + a
        x = x + self.mlp(self.ln2(x))
        return x, new_cache

class LegalTransformerLM(nn.Module):
    # It takes vocab_size (vocabulary size), dim (model size), n_layers (number of layers), n_heads (number of heads), max_len (max length), dropout (dropout rate) and tie (tie embeddings) as input, builds the full Legal Transformer model and returns nothing.
    def __init__(self, vocab_size, dim=896, n_layers=24, n_heads=14, max_len=1024,
                 dropout=0.1, tie=True):
        super().__init__()
        assert dim % n_heads == 0, "dim must be divisible by n_heads"
        self.dim = dim
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.max_len = max_len

        self.tok_emb = nn.Embedding(vocab_size, dim)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([TransformerBlock(dim, n_heads, dropout) for _ in range(n_layers)])
        self.norm_f = nn.LayerNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)
        if tie:

            self.lm_head.weight = self.tok_emb.weight

        cos, sin = build_rope_cache(self.head_dim, max_len, device=torch.device("cpu"))
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)

        residual_scale = (2 * n_layers) ** -0.5
        for blk in self.blocks:
            nn.init.normal_(blk.attn.wo.weight, mean=0.0, std=0.02 * residual_scale)
            nn.init.normal_(blk.mlp.fc2.weight, mean=0.0, std=0.02 * residual_scale)

    # It sets the starting values of the weights and returns nothing.
    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    # It takes input_ids (token ids), kv_cache (saved keys and values) and pos_offset (position offset) as input, runs the model and returns next-token scores and the cache.
    def forward(self, input_ids, kv_cache=None, pos_offset=0):

        B, T = input_ids.shape
        x = self.drop(self.tok_emb(input_ids))
        new_cache = []
        for i, blk in enumerate(self.blocks):
            layer_cache = kv_cache[i] if kv_cache is not None else None
            x, lc = blk(x, self.rope_cos, self.rope_sin, pos_offset, layer_cache)
            new_cache.append(lc)
        logits = self.lm_head(self.norm_f(x))
        return logits, new_cache

class LegalTokenizer(LegalByteBPETokenizer):

    # It returns the vocabulary size.
    @property
    def vocab_size(self):
        return len(self.vocab)

    # It returns the padding token id.
    @property
    def pad_id(self):
        return self.pad_token_id

    # It returns the end-of-text token id.
    @property
    def eos_id(self):
        return self.eos_token_id

    bos_id = None

    # It takes text as input, converts text into token ids and remember where each token is in the text and returns (token ids, character offsets).
    def encode_with_offsets(self, text):

        ids, offs = [], []
        for m in _PRETOK_RE.finditer(text):
            chunk = m.group()
            base = m.start()
            b2c = []
            for ci, ch in enumerate(chunk):
                b2c.extend([base + ci] * len(ch.encode("utf-8")))
            sym = tuple(_BYTE_ENCODER[b] for b in chunk.encode("utf-8"))
            pos = 0
            for piece in self._bpe(sym):
                nb = len(piece)
                ids.append(self.vocab[piece])
                offs.append((b2c[pos], b2c[pos + nb - 1] + 1))
                pos += nb
        return ids, offs

# It takes tok_path (tokenizer file path) as input, loads the tokenizer from its JSON file and returns a tokenizer.
def load_tokenizer(tok_path):
    return LegalTokenizer.load(tok_path)

# It takes path (file path) as input, finds the real checkpoint file to load and returns (file path, temporary folder or None).
def _resolve_ckpt(path):

    if os.path.isfile(path):
        return path, None
    if not os.path.isdir(path):
        raise FileNotFoundError(path)
    root = None
    for dp, _dn, fn in os.walk(path):
        if "data.pkl" in fn:
            root = dp
            break
    if root is None:
        raise FileNotFoundError(f"{path} is a folder without data.pkl")
    prefix = os.path.splitext(os.path.basename(os.path.normpath(root)))[0] or "archive"
    tmpdir = tempfile.mkdtemp(prefix="legal_ckpt_")
    out = os.path.join(tmpdir, "ckpt.pt")
    with zipfile.ZipFile(out, "w", zipfile.ZIP_STORED) as z:
        for dp, _dn, fn in os.walk(root):
            for f in sorted(fn):
                full = os.path.join(dp, f)
                z.write(full, arcname=prefix + "/" + os.path.relpath(full, root).replace(os.sep, "/"))
    return out, tmpdir

class Backbone:

    # It takes model, cfg (settings) and rope_len (rotary table length) as input, wraps the pre-trained model with simple helper methods and returns nothing.
    def __init__(self, model, cfg, rope_len):
        self.model = model
        self.config = dict(cfg)
        self.d_model = int(cfg["dim"])
        self.max_len = TRAINED_SEQ_TOKENS
        self.rope_len = rope_len
        self.n_params = sum(p.numel() for p in model.parameters())
        self.bidirectional = False
        self.causal = True

    # It takes flag (on/off) as input, turns the causal mask off or on and returns nothing.
    def set_bidirectional(self, flag: bool):

        self.bidirectional = bool(flag)
        self.causal = not self.bidirectional
        for blk in self.model.blocks:
            blk.attn.bidirectional = self.bidirectional

    # It takes input_ids (token ids) and attention_mask (mask of real tokens) as input, returns the hidden vector of every token.
    def hidden(self, input_ids, attention_mask=None):

        m = self.model
        B, T = input_ids.shape
        if T > self.rope_len:
            raise ValueError(f"sequence length {T} > RoPE table {self.rope_len}; "
                             f"load_backbone(..., max_len>={T}) (trained only up to {self.max_len})")
        kpm = None
        if attention_mask is not None:
            am = attention_mask.to(input_ids.device).bool()
            if not bool(am.all()):
                kpm = am
        attns = [blk.attn for blk in m.blocks]
        for a in attns:
            a.key_pad_mask = kpm
        try:
            x = m.drop(m.tok_emb(input_ids))
            for blk in m.blocks:
                x, _ = blk(x, m.rope_cos, m.rope_sin, 0, None)
            return m.norm_f(x)
        finally:
            for a in attns:
                a.key_pad_mask = None

    # It returns the names of the linear layers inside the blocks.
    def linear_names(self):

        return [n for n, mod in self.model.named_modules()
                if n.startswith("blocks.") and isinstance(mod, nn.Linear)]

    # It returns the list of model blocks.
    def block_modules(self):
        return list(self.model.blocks)

# It takes ckpt_path (checkpoint path), device, max_len (max length) and config (model settings) as input, loads the pre-trained model from its checkpoint and returns the backbone ready to use.
def load_backbone(ckpt_path, device="cpu", max_len=None, config=None):
    path, tmp = _resolve_ckpt(ckpt_path)
    try:
        pkg = torch.load(path, weights_only=False, map_location="cpu")
    finally:
        if tmp:
            shutil.rmtree(tmp, ignore_errors=True)
    if isinstance(pkg, dict) and "state_dict" in pkg:
        sd, cfg = pkg["state_dict"], dict(pkg.get("config") or {})
    else:
        sd, cfg = pkg, {}
    if config:
        cfg.update(config)
    sd = {(k[len("module."):] if k.startswith("module.") else k): v for k, v in sd.items()}
    if "vocab_size" not in cfg or "dim" not in cfg:
        w = sd["tok_emb.weight"]
        cfg.setdefault("vocab_size", w.shape[0]); cfg.setdefault("dim", w.shape[1])
        cfg.setdefault("n_layers", 1 + max(int(k.split(".")[1]) for k in sd if k.startswith("blocks.")))
    for k in ("n_heads", "n_layers"):
        if k not in cfg:
            raise ValueError(f"checkpoint config lacks {k!r}; pass config={{'{k}': ...}}")
    cfg.setdefault("dropout", 0.1)
    cfg.setdefault("tied", "lm_head.weight" not in sd or sd["lm_head.weight"].data_ptr() == sd["tok_emb.weight"].data_ptr())
    rope_len = max(int(cfg.get("max_len", NOTEBOOK_ROPE_LEN)), int(max_len or 0))
    model = LegalTransformerLM(cfg["vocab_size"], dim=cfg["dim"], n_layers=cfg["n_layers"],
                               n_heads=cfg["n_heads"], max_len=rope_len,
                               dropout=cfg["dropout"], tie=cfg["tied"])
    missing, unexpected = model.load_state_dict(sd, strict=False)
    if cfg["tied"]:
        missing = [k for k in missing if k != "lm_head.weight"]
    if missing or unexpected:
        print("!" * 70, f"\n!!! STATE_DICT MISMATCH  missing={missing[:10]} (n={len(missing)})  "
              f"unexpected={unexpected[:10]} (n={len(unexpected)})\n" + "!" * 70)
        raise RuntimeError("checkpoint does not match Legal-Transformer architecture")
    if cfg["tied"]:
        assert model.lm_head.weight is model.tok_emb.weight
    model.to(device)
    bb = Backbone(model, cfg, rope_len)
    bb.max_len = min(TRAINED_SEQ_TOKENS, rope_len)
    return bb

Writing arch_legal_transformer.py


In [2]:
# Import libraries and file finder helper
import os, re, json, glob, math, time, zipfile, numpy as np, torch, torch.nn as nn, torch.nn.functional as F, torch.utils.checkpoint
DEV, K, OUT = "cuda" if torch.cuda.is_available() else "cpu", 7, "/kaggle/working" if os.path.isdir("/kaggle/working") else "./out"
os.makedirs(OUT, exist_ok=True)
# It takes *patterns (any arguments) as input, finds the right input file to use and returns a path of the chosen file.
def pick(*patterns):
    for p in patterns:
        h = sorted(x for x in glob.glob(f"/kaggle/input/**/{p}", recursive=True) if os.path.isfile(x) and "trainable" not in x)
        if h: print("using", h[0]); return h[0]
    raise FileNotFoundError(f"none of {patterns} found in /kaggle/input")

In [3]:
# Load Legal Transformer and set LoRMA settings
import arch_legal_transformer as A
bb = A.load_backbone(pick("legal_transformer*.pt", "*transformer*.pt")); tok = A.load_tokenizer(pick("tokenizer_legal_transformer*.json", "*tokenizer*transformer*.json"))
pid = tok.pad_id or 0
NAME, TECH = "p7_lt_lorma", "lorma"
c = dict(lr=0.0002, epochs=2, hours=4.0, bs=8, chunk=512, kinf=4, ktrain=2, tau=0.05, hard=40, r=16, alpha=32, ar=64, teacher="", beta=0.5, teach=1, seed=0)

using /kaggle/input/datasets/mannpatel2315/legal-trans-final-pt-p2/legal_transformer_random_moredata_best_random_90_10.pt
using /kaggle/input/datasets/mannpatel2315/legal-trans-tokenizer-p2/tokenizer_legal_transformer_random_moredata_random_90_10.json


In [4]:
# Load queries and candidate cases
# It takes path (file path) as input, loads queries or candidates from a JSON file and returns a list of (id, text, relevant ids).
def read(path):
    r = json.load(open(path, encoding="utf-8"))
    if isinstance(r, dict): r = [dict(v, id=k) if isinstance(v, dict) else dict(id=k, text=v) for k, v in r.items()]
    print("fields:", list(r[0]))
    return [(str(d["id"]), " ".join(d["text"]) if isinstance(d["text"], list) else d["text"], [str(x) for x in d.get("relevant_candidates", [])]) for d in r]
# It takes split (split name) as input, loads the queries and candidates of one split and returns (queries, candidates).
def load_split(split):
    out = []
    for n in (split.lower() + ".json", split.lower() + "_candidates.json"):
        f = [x for x in glob.glob(f"/kaggle/input/**/{n}", recursive=True) if "phase_7" in x.lower()]
        if not f:
            from huggingface_hub import hf_hub_download, list_repo_files, login
            try:
                from kaggle_secrets import UserSecretsClient; login(UserSecretsClient().get_secret("HF_TOKEN"))
            except Exception: login(os.environ.get("HF_TOKEN"))
            repo = "Exploration-Lab/CS-781-Capstone"; f = [hf_hub_download(repo, x, repo_type="dataset") for x in list_repo_files(repo, repo_type="dataset") if x.lower().endswith("/" + n) and "phase_7" in x.lower()]
        out.append(read(f[0]))
    return out
D = {s: load_split(s) for s in ("Train", "Dev", "Test")}
print({s: (len(q), len(c_)) for s, (q, c_) in D.items()}, "(queries, candidates)")

Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

fields: ['text', 'relevant_candidates', 'id']


Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

fields: ['text', 'id']


... (6 outputs left out)

fields: ['text', 'id']


Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

fields: ['text', 'id']
{'Train': (827, 4320), 'Dev': (118, 1023), 'Test': (237, 1727)} (queries, candidates)


In [5]:
# Lexical scores for every query and candidate
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from scipy.sparse import csr_matrix
# It takes qs (queries) and cs (candidates) as input, computes BM25 and TF-IDF scores and returns (BM25 matrix, TF-IDF matrix).
def lexical(qs, cs):
    tv = TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2), min_df=2, max_features=300000).fit(cs)
    tfidf = (tv.transform(qs) @ tv.transform(cs).T).toarray()
    cv = CountVectorizer(min_df=2).fit(cs); C = cv.transform(cs).tocoo(); n = C.shape[0]
    df, dl = np.bincount(C.col, minlength=C.shape[1]), np.bincount(C.row, weights=C.data, minlength=n)
    w = C.data * 2.2 / (C.data + 1.2 * (.25 + .75 * dl[C.row] / dl.mean())) * np.log(1 + (n - df[C.col] + .5) / (df[C.col] + .5))
    Q = cv.transform(qs); Q.data = np.log1p(Q.data)
    return (Q @ csr_matrix((w, (C.row, C.col)), shape=C.shape).T).toarray(), tfidf
Lx = {s: lexical([q[1] for q in qs], [d[1] for d in cs]) for s, (qs, cs) in D.items()}
lex = Lx

In [6]:
# Scoring metric (Micro-F1@7) and score mixing
z = lambda S: (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-9)
# It takes S (score matrix), qs (queries) and cs (candidates) as input, gets the top 7 candidates for each query and returns indices of the top 7 candidates per query.
def top7(S, qs, cs):
    S = np.where(np.array([d[0] for d in cs])[None] == np.array([q[0] for q in qs])[:, None], -1e9, S); return np.argsort(-S, 1)[:, :K]
# It takes S (score matrix), qs (queries) and cs (candidates) as input, computes Micro-F1 at 7 and returns the Micro-F1@7.
def micro_f1(S, qs, cs):
    tp = sum(len({cs[j][0] for j in r} & set(q[2])) for r, q in zip(top7(S, qs, cs), qs)); p, r = tp / (K * len(qs)), tp / sum(len(q[2]) for q in qs); return 2 * p * r / (p + r + 1e-9)
# It takes S (score matrix), lam (strength) and f (fraction) as input, lowers the scores of candidates that match too many queries and returns a adjusted score matrix.
def hub(S, lam, f):
    return S - lam * np.sort(S, 0)[-max(1, int(f * len(S))):].mean(0)
# It takes S (score matrix), t (temperature) and it (number of rounds) as input, balances scores so each candidate fits few queries and returns a balanced score matrix.
def sink(S, t, it=3):
    P = np.exp((S - S.max()) / t) + 1e-30
    for _ in range(it): P /= P.sum(0, keepdims=True); P /= P.sum(1, keepdims=True)
    return np.log(P)
# It takes model, lex (BM25 and TF-IDF scores), a (model weight), b (BM25 weight) and post (post-processing setting) as input, mixes model, BM25 and TF-IDF scores and returns a mixed score matrix.
def blend(model, lex, a, b, post):
    S = a * z(model) + (1 - a) * (b * z(lex[0]) + (1 - b) * z(lex[1])); return hub(S, *post[1:]) if post[0] == "hub" else sink(S, post[1])

In [7]:
# Fine-tuning methods: LoRMA, adapter, full
class LoRMA(nn.Module):
    # It takes base (base linear layer), r (rank) and alpha (scale of the low-rank update) as input, creates the small LoRMA matrices around a linear layer and returns nothing.
    def __init__(s, base, r, alpha):
        super().__init__(); s.base, s.s = base, alpha / r; s.A = nn.Parameter(torch.randn(r, base.out_features) / math.sqrt(r)); s.B = nn.Parameter(torch.zeros(base.out_features, r))
    # It takes x (input tensor) as input, runs the linear layer with the LoRMA update and returns the output of the updated layer.
    def forward(s, x): y = F.linear(x, s.base.weight.to(x.dtype)); y = y + s.s * F.linear(F.linear(y, s.A.to(x.dtype)), s.B.to(x.dtype)); return y if s.base.bias is None else y + s.base.bias.to(x.dtype)
class Adapter(nn.Module):
    # It takes blk (model block), d (model size) and r (adapter size) as input, creates a small adapter that starts as no change and returns nothing.
    def __init__(s, blk, d, r): super().__init__(); s.blk, s.dn, s.up = blk, nn.Linear(d, r), nn.Linear(r, d); nn.init.zeros_(s.up.weight); nn.init.zeros_(s.up.bias)
    # It takes x (input tensor) as input, runs the block and adds the adapter output and returns output with the adapter added.
    def forward(s, x): x = s.blk(x); return x + s.up(F.gelu(s.dn(x)))
class Ck(nn.Module):
    # It takes b (block) as input, wraps a block to save GPU memory and returns nothing.
    def __init__(s, b): super().__init__(); s.b = b
    # It returns the attention layer of the wrapped block.
    @property
    def attn(s): return s.b.attn
    # It takes *a (any arguments) as input, runs the block while saving memory and returns block output.
    def forward(s, *a): return torch.utils.checkpoint.checkpoint(s.b, *a, use_reentrant=False) if s.training and torch.is_grad_enabled() else s.b(*a)
# It takes bb (pre-trained backbone) and tech (fine-tuning method) as input, prepares the model for the chosen fine-tuning method and returns the trainable weights.
def adapt(bb, tech):
    lm = bb.model
    if tech != "full":
        for p in lm.parameters(): p.requires_grad_(False)
    if tech == "lorma":
        for n in bb.linear_names():
            par = lm; *path, leaf = n.split(".")
            for q in path: par = getattr(par, q)
            setattr(par, leaf, LoRMA(getattr(par, leaf), c["r"], c["alpha"]))
    if tech == "adapter":
        for i, b in enumerate(lm.blocks):
            if type(b).__name__ == "FeedForward": lm.blocks[i] = Adapter(b, bb.d_model, c["ar"])
        for p in lm.norm_f.parameters(): p.requires_grad_(True)
    if hasattr(bb, "ck"): bb.ck = True
    else: lm.blocks = nn.ModuleList(Ck(b) for b in lm.blocks)
    return [p for p in lm.parameters() if p.requires_grad]

In [8]:
# Convert documents to tokens and batches
# It takes texts as input, tokenizes documents, keeping the first part and returns a list of token arrays.
def tokenize(texts):
    n = c["chunk"] * c["kinf"]; return [np.array(tok.encode(t[:n * 6])[:n] or [0], np.int32) for t in texts]
# It takes rows (token lists) as input, pads token arrays and makes a mask and returns (padded ids, mask).
def pad(rows):
    L = torch.tensor([len(r) for r in rows]); x = torch.full((len(rows), int(L.max())), pid, dtype=torch.long)
    for i, r in enumerate(rows): x[i, :len(r)] = torch.as_tensor(r, dtype=torch.long)
    return x.to(DEV), (torch.arange(x.size(1))[None] < L[:, None]).to(DEV)

In [9]:
# Turn documents into vectors and compare them
# It takes docs (token arrays of the documents) and rng (random generator) as input, turns documents into vectors and returns one vector per document.
def doc_vectors(docs, rng=None):
    segs = []
    for i, d in enumerate(docs):
        n = max(1, math.ceil(len(d) / c["chunk"])); js = rng.choice(n, min(c["ktrain"], n), replace=False) if rng is not None else range(n)
        segs += [(i, d[j * c["chunk"]:(j + 1) * c["chunk"]]) for j in js]
    x, m = pad([s for _, s in segs]); ix = torch.tensor([i for i, _ in segs], device=DEV)
    with torch.autocast("cuda", dtype=torch.float16, enabled=DEV == "cuda"):
        h = bb.hidden(x, m); w = m[..., None].to(h.dtype); h = ((h * w).sum(1) / w.sum(1).clamp(min=1)).float()
    return torch.zeros(len(docs), h.size(1), device=DEV).index_add(0, ix, h) / torch.bincount(ix, minlength=len(docs))[:, None]
# It takes Xq (query token ids), Xc (candidate token ids) and bs (batch size) as input, returns a query x candidate similarity matrix; it is the cosine similarity between all queries and candidates.
@torch.no_grad()
def scores(Xq, Xc, bs=24):
    bb.model.eval(); vec = lambda X: torch.cat([doc_vectors(X[i:i + bs]).cpu() for i in range(0, len(X), bs)])
    q, k = vec(Xq), vec(Xc); mu = k.mean(0); return (F.normalize(q - mu, dim=-1) @ F.normalize(k - mu, dim=-1).T).numpy()

In [10]:
# Training loss (contrastive + distillation)
# It takes qi (query indices), di (candidate indices) and rng (random generator) as input, returns a loss value; it is the contrastive loss, plus distillation if a teacher is given.
def batch_loss(qi, di, rng):
    B = len(qi); v = doc_vectors([Xq[i] for i in qi] + [Xc[j] for j in di], rng); v = F.normalize(v - v.mean(0), dim=-1); lg = v[:B] @ v[B:].T / c["tau"]
    lg = lg - 1e4 * torch.tensor([[cj in gold[q] and k != i for k, cj in enumerate(di)] for i, q in enumerate(qi)], device=DEV)
    loss = F.cross_entropy(lg, torch.arange(B, device=DEV))
    if T is not None:
        t = torch.tensor(np.array([[T[q, j] for j in di] for q in qi]), device=DEV) / c["tau"] - (lg < -5e3) * 1e4
        loss = (1 - c["beta"]) * loss + c["beta"] * F.kl_div(F.log_softmax(lg, 1), F.softmax(t, 1), reduction="batchmean")
    return loss

In [11]:
# Training loop
# It takes ps (parameters) and evaluate (evaluation function) as input, trains the model and keeps the best epoch and returns training time in seconds.
def fit(ps, evaluate):
    rng = np.random.default_rng(c["seed"]); opt = torch.optim.AdamW(ps, lr=c["lr"], weight_decay=0.01); sc = torch.amp.GradScaler("cuda", enabled=DEV == "cuda")
    spe = len(pairs) // c["bs"]; total, t0 = c["epochs"] * spe, time.time(); best, bst = evaluate(), {k: p.detach().cpu().clone() for k, p in bb.model.named_parameters() if p.requires_grad}; print(f"   epoch 0 (no fine-tuning) dev {best:.4f}"); bb.model.train()
    for step in range(1, total + 1):
        for g in opt.param_groups: g["lr"] = c["lr"] * min(1, step / 20) * 0.5 * (1 + math.cos(math.pi * step / total))
        b = [pairs[i] for i in rng.choice(len(pairs), c["bs"], replace=False)]; qi = [x[0] for x in b]; di = [x[1] for x in b] + [int(rng.choice(hard[q])) for q in qi]
        opt.zero_grad(set_to_none=True); loss = batch_loss(qi, di, rng); sc.scale(loss).backward(); sc.unscale_(opt); nn.utils.clip_grad_norm_(ps, 1.0); sc.step(opt); sc.update()
        if step % 20 == 0: print(f"step {step}/{total} loss {loss.item():.4f} | {(time.time() - t0) / 60:.1f} min", flush=True)
        if step % spe == 0 or time.time() - t0 > c["hours"] * 3600:
            v = evaluate(); print(f"   dev micro-F1@7 (model + lexical + hubness) {v:.4f}", flush=True); bb.model.train()
            if v > best: best, bst = v, {k: p.detach().cpu().clone() for k, p in bb.model.named_parameters() if p.requires_grad}
            if time.time() - t0 > c["hours"] * 3600: break
    bb.model.load_state_dict(bst, strict=False); return time.time() - t0

In [12]:
# Prepare training pairs and hard negatives
X = {s: (tokenize([q[1] for q in qs]), tokenize([d[1] for d in cs])) for s, (qs, cs) in D.items()}
(qs, cs), (Xq, Xc) = D["Train"], X["Train"]; ix = {d[0]: j for j, d in enumerate(cs)}
gold = [{ix[r] for r in q[2] if r in ix} for q in qs]; pairs = [(i, j) for i, g in enumerate(gold) for j in g]
lx = z(Lx["Train"][0]) + z(Lx["Train"][1]); hard = [[j for j in np.argsort(-lx[i])[:c["hard"] + len(gold[i])] if j not in gold[i]] for i in range(len(qs))]
T = np.load(pick(c["teacher"]))["train_model"] if c["teacher"] else None; assert T is None or T.shape == (len(qs), len(cs)), "teacher scores do not match the training data"
print(f"{len(qs)} train queries, {len(pairs)} positive pairs, distillation: {T is not None}")

827 train queries, 5308 positive pairs, distillation: False


In [13]:
# Fine-tune the model
dq, dc = D["Dev"]; ps = adapt(bb, TECH); bb.model.to(DEV); ntr, nall = sum(p.numel() for p in ps), sum(p.numel() for p in bb.model.parameters())
print(f"{TECH}: {ntr / 1e6:.2f}M trainable of {nall / 1e6:.1f}M parameters")
ref = lambda: micro_f1(blend(scores(*X["Dev"]), lex["Dev"], .5, .5, ("hub", 1, .08)), dq, dc)
f1_before = micro_f1(scores(*X["Dev"]), dq, dc); print(f"dev micro-F1@7 with the Phase-1 representations only (raw): {f1_before:.4f}")
secs = fit(ps, ref)

lorma: 6.97M trainable of 278.1M parameters
dev micro-F1@7 with the Phase-1 representations only (raw): 0.2268
   epoch 0 (no fine-tuning) dev 0.3824
step 20/1326 loss 5.2027 | 4.9 min
step 40/1326 loss 6.7224 | 7.1 min
step 60/1326 loss 4.2176 | 9.2 min
step 120/1326 loss 2.5675 | 15.6 min
step 140/1326 loss 2.1407 | 17.7 min
step 160/1326 loss 2.7562 | 19.8 min
step 180/1326 loss 2.4301 | 21.9 min
step 200/1326 loss 2.8195 | 24.1 min
step 220/1326 loss 1.5214 | 26.2 min
step 240/1326 loss 1.8888 | 28.3 min
step 260/1326 loss 2.0731 | 30.5 min
step 280/1326 loss 1.6341 | 32.9 min
step 300/1326 loss 1.5788 | 35.2 min
step 320/1326 loss 2.8159 | 37.5 min
step 340/1326 loss 1.8715 | 39.9 min
step 360/1326 loss 1.6219 | 42.2 min
step 380/1326 loss 1.7197 | 44.3 min
step 400/1326 loss 1.4288 | 46.5 min
step 420/1326 loss 1.4196 | 48.6 min
step 440/1326 loss 1.9707 | 50.7 min
step 460/1326 loss 1.5322 | 52.8 min
step 480/1326 loss 0.8212 | 55.0 min
step 500/1326 loss 1.5507 | 57.1 min
step 

In [14]:
# Compute and save scores for all splits
S = {s: scores(*X[s]) for s in ("Dev", "Test")}
if c["teach"]: S["Train"] = scores(Xq, Xc)
np.savez_compressed(f"{OUT}/{NAME}_scores.npz", **{f"{s.lower()}_model": v for s, v in S.items()}, **{f"{s.lower()}_{k}": Lx[s][i] for s in Lx for i, k in enumerate(("bm25", "tfidf"))},
                    **{f"{s.lower()}_{k}": np.array([d[0] for d in D[s][i]]) for s in D for i, k in enumerate(("qids", "cids"))})

In [15]:
# Pick best settings on dev and write submissions
posts = [("hub", l, k) for l in (0, .5, 1, 1.5, 2, 3) for k in (.02, .08, .2)] + [("sink", t) for t in (.2, .35, .5, 1)]
res = {(a_, b_, p): micro_f1(blend(S["Dev"], lex["Dev"], a_, b_, p), dq, dc) for a_ in (1, .7, .5, .35, .25, .15, .08, 0) for b_ in (.5, 1) for p in posts}
f1_lex = micro_f1(z(lex["Dev"][0]) + z(lex["Dev"][1]), dq, dc); f1_raw = micro_f1(S["Dev"], dq, dc); print(f"dev micro-F1@7 | lexical only {f1_lex:.4f} | model only {f1_raw:.4f} | model before fine-tuning {f1_before:.4f}")
tq, tc = D["Test"]; top = sorted(res, key=res.get, reverse=True); print("best settings:", [(k, round(res[k], 4)) for k in top[:5]])
for a_, b_, p in top[:4]:
    pred = {q[0]: [tc[j][0] for j in r] for q, r in zip(tq, top7(blend(S["Test"], lex["Test"], a_, b_, p), tq, tc))}
    assert len(pred) == len(tq) and all(len(set(v)) == K for v in pred.values())
    f = f"{OUT}/{NAME}_dev{res[(a_, b_, p)]:.4f}_m{a_}_b{b_}_{p[0]}{'_'.join(map(str, p[1:]))}_submission.zip"; zipfile.ZipFile(f, "w", zipfile.ZIP_DEFLATED).writestr("prediction.json", json.dumps(pred)); print(f)
json.dump(dict(name=NAME, technique=TECH, trainable_M=ntr / 1e6, total_M=nall / 1e6, train_seconds=secs, dev_model_only=f1_raw, dev_before_ft=f1_before, dev_lexical=f1_lex, grid={str(k): v for k, v in res.items()}, settings=c), open(f"{OUT}/{NAME}_summary.json", "w"), indent=1)

dev micro-F1@7 | lexical only 0.3180 | model only 0.2657 | model before fine-tuning 0.2268
best settings: [((0, 0.5, ('hub', 1, 0.08)), 0.448), ((0.08, 0.5, ('hub', 1, 0.08)), 0.4458), ((0.15, 0.5, ('hub', 1, 0.08)), 0.4447), ((0, 0.5, ('sink', 0.5)), 0.4436), ((0.08, 0.5, ('sink', 0.5)), 0.4425)]
/kaggle/working/p7_lt_lorma_dev0.4480_m0_b0.5_hub1_0.08_submission.zip
/kaggle/working/p7_lt_lorma_dev0.4458_m0.08_b0.5_hub1_0.08_submission.zip
/kaggle/working/p7_lt_lorma_dev0.4447_m0.15_b0.5_hub1_0.08_submission.zip
/kaggle/working/p7_lt_lorma_dev0.4436_m0_b0.5_sink0.5_submission.zip


In [16]:
# Zip all outputs
with zipfile.ZipFile(f"{OUT}/ALL_outputs.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(glob.glob(f"{OUT}/*")):
        if os.path.isfile(p) and not p.endswith("ALL_outputs.zip"): zf.write(p, os.path.basename(p))